In [0]:
# ============================================================
# 0. CONFIG PATH
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

# ============================================================
# 1. IMPORTS
# ============================================================

import json

import pyspark.sql.functions as F
from pyspark.sql.window import Window

from common_utils.logging import get_logger
from common_utils.generic_functions import safe_cast


# ============================================================
# 2. LOGGER
# ============================================================

logger = get_logger("fact_sales_gold")


# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info("Reading Gold configuration from %s",config_path)

with open(config_path, "r") as f:
    config = json.load(f)


logger.info("Gold configuration loaded successfully")


# ============================================================
# 1. SOURCE / TARGET CONFIGURATION
# ============================================================

source_config = config["source"]
target_config = config["target"]

source_name = source_config["source_name"]

sales_orders_table = source_config["tables"]["sales_orders"]
order_products_table = source_config["tables"]["order_products"]
promotions_table = source_config["tables"]["promotions"]

target_table = target_config["target_table"]


logger.info("[%s] Source tables configured", source_name)

logger.info("[%s] Target table: %s",source_name,target_table)


# ============================================================
# 2. READ SILVER TABLES
# ============================================================

logger.info("[%s] Reading sales_orders from %s",source_name,sales_orders_table)
sales_orders_df = spark.read.table(sales_orders_table)

logger.info("[%s] Reading order_products from %s",source_name,order_products_table)
order_products_df = spark.read.table(order_products_table)

logger.info("[%s] Reading promotions from %s",source_name,promotions_table)
promotions_df = spark.read.table(promotions_table)

logger.info("[%s] All Silver tables successfully loaded",source_name)


# ============================================================
# 6. TRANSFORM PROMOTIONS
# ============================================================

def prepare_promotions(df, config):

    logger.info("[%s] Preparing promotions",source_name)
    # --------------------------------------------------------
    # Keep only columns required for fact table
    # --------------------------------------------------------
    df = df.select(
        "order_number",
        "promo_product_id",
        "discount",
        "promo_quantity"
    )


    # --------------------------------------------------------
    # Cast data types
    # --------------------------------------------------------
    df = safe_cast(
        df,
        {
            "order_number": "long",
            "promo_product_id": "string",
            "discount": "double",
            "promo_quantity": "long"
        }
    )


    # --------------------------------------------------------
    # Remove duplicate promotion records
    #
    # We want ONE promotion record per:
    #
    # order_number + product_id
    #
    # Otherwise joining promotions to order_products
    # could create duplicate fact rows.
    # --------------------------------------------------------

    df = (
        df
        .groupBy(
            "order_number",
            "promo_product_id"
        )
        .agg(
            F.max("discount").alias("discount"),
            F.max("promo_quantity").alias("promo_quantity")
        )
    )


    logger.info("[%s] Promotions prepared successfully",source_name)

    return df


# ============================================================
# 7. TRANSFORM FACT SALES
# ============================================================

def transform_fact_sales(
    sales_orders_df,
    order_products_df,
    promotions_df,
    config
):

    logger.info("[%s] Starting fact_sales transformation",source_name)


    # ========================================================
    # 7.1 PREPARE ORDERS
    # ========================================================

    logger.info(
        "[%s] Preparing sales_orders",
        source_name
    )

    orders = (
        sales_orders_df
        .select(
            "order_number",
            "customer_id",
            "order_timestamp"
        )
        .withColumn(
            "order_date_key",
            F.date_format(
                F.to_timestamp(
                    F.col("order_timestamp")
                ),
                config["transform"]["order_date_key_format"]
            ).cast("int")
        )
        .select(
            "order_number",
            "customer_id",
            "order_date_key"
        )
    )


    # ========================================================
    # 7.2 PREPARE ORDER PRODUCTS
    # ========================================================

    logger.info(
        "[%s] Preparing order_products",
        source_name
    )

    products = (
        order_products_df
        .select(
            "order_number",
            "customer_id",
            F.col("id").alias("product_id"),
            F.col("qty").alias("qty"),
            F.col("price").alias("unit_price")
        )
    )


    # ========================================================
    # 7.3 PREPARE PROMOTIONS
    # ========================================================

    promotions = prepare_promotions(promotions_df,config)


    # ========================================================
    # 7.4 JOIN ORDERS + PRODUCTS
    # ========================================================

    logger.info("[%s] Joining sales_orders with order_products",source_name)

    fact = (
        products.alias("p")
        .join(
            orders.alias("o"),
            F.col("p.order_number")
            == F.col("o.order_number"),
            "left"
        )
    )


    # ========================================================
    # 7.5 JOIN PROMOTIONS
    # ========================================================

    logger.info("[%s] Joining promotions",source_name)

    fact = (
        fact
        .join(
            promotions.alias("pr"),
            (
                (F.col("p.order_number")
                 == F.col("pr.order_number"))
                &
                (F.col("p.product_id")
                 == F.col("pr.promo_product_id"))
            ),
            "left"
        )
    )


    # ========================================================
    # 7.6 SELECT FACT COLUMNS
    # ========================================================

    logger.info("[%s] Selecting fact_sales columns",source_name)

    fact = fact.select(

        # ----------------------------------------------------
        # Keys
        # ----------------------------------------------------

        F.col("p.order_number")
            .alias("order_number"),

        F.col("o.customer_id")
            .alias("customer_id"),

        F.col("p.product_id")
            .alias("product_id"),

        F.col("o.order_date_key")
            .alias("order_date_key"),


        # ----------------------------------------------------
        # Measures
        # ----------------------------------------------------

        F.col("p.qty")
            .alias("qty"),

        F.col("p.unit_price")
            .alias("unit_price"),


        # ----------------------------------------------------
        # Promotion information
        # ----------------------------------------------------

        F.coalesce(
            F.col("pr.discount"),
            F.lit(
                config["transform"]["default_discount"]
            )
        ).alias("discount"),

        F.coalesce(
            F.col("pr.promo_quantity"),
            F.lit(
                config["transform"]["default_promo_quantity"]
            )
        ).alias("promo_quantity")
    )


    # ========================================================
    # 7.7 DERIVE TOTAL AMOUNT
    # ========================================================

    logger.info(
        "[%s] Calculating total_amount",
        source_name
    )

    fact = fact.withColumn(
        "total_amount",
        F.round(
            F.col("qty")
            * F.col("unit_price")
            * (
                F.lit(1)
                - F.col("discount")
            ),
            2
        )
    )


    # ========================================================
    # 7.8 CAST FINAL COLUMNS
    # ========================================================

    logger.info("[%s] Applying final data types",source_name)

    fact = safe_cast(fact,config.get("cast_columns", {}))


    # ========================================================
    # 7.9 SELECT FINAL COLUMNS
    # ========================================================

    fact = fact.select(
        *config["select_columns"]
    )


    # ========================================================
    # 7.10 REMOVE DUPLICATES
    # ========================================================

    logger.info(
        "[%s] Removing duplicate fact records",
        source_name
    )

    fact = fact.dropDuplicates(
        [
            "order_number",
            "product_id"
        ]
    )


    logger.info(
        "[%s] fact_sales transformation completed",
        source_name
    )


    return fact




# ============================================================
# 8. CREATE FACT SALES
# ============================================================

logger.info("[%s] Starting Gold fact_sales creation",source_name)

fact_sales = transform_fact_sales(
    sales_orders_df=sales_orders_df,
    order_products_df=order_products_df,
    promotions_df=promotions_df,
    config=config
)




# ============================================================
# 9. VALIDATE FACT DATA
# ============================================================

logger.info("[%s] Validating fact_sales",source_name)
record_count = fact_sales.count()
logger.info("[%s] fact_sales record count: %s",source_name,record_count)


if record_count == 0:
    raise ValueError(
        "fact_sales contains zero records"
    )


# Check for duplicate grain

duplicate_count = (
    fact_sales
    .groupBy(
        "order_number",
        "product_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


logger.info("[%s] Duplicate fact grain count: %s",source_name,duplicate_count)


if duplicate_count > 0:
    raise ValueError(
        "Duplicate order_number + product_id "
        "records detected in fact_sales"
    )

# ============================================================
# 10. DISPLAY FACT
# ============================================================

fact_sales.show()

# ============================================================
# 11. CREATE GOLD SCHEMA
# ============================================================

logger.info("[%s] Creating Gold schema if required",source_name)

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    retaildataplatform.gold
""")

# ============================================================
# 12. WRITE GOLD TABLE
# ============================================================

logger.info("[%s] Writing fact_sales to %s",source_name,target_table)


(
    fact_sales
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(target_table)
)


logger.info("[%s] Gold table %s written successfully",source_name,target_table)




# ============================================================
# 13. FINAL VALIDATION
# ============================================================

logger.info("[%s] Running final Gold table validation",source_name)
final_count = (spark.read.table(target_table).count())
logger.info("[%s] Final Gold record count: %s",source_name,final_count)
logger.info("[%s] Gold fact_sales pipeline completed successfully",source_name)